# 🏁 UIT-CAR-RACING: Training YOLO11n-seg cho Sa hình Ban đêm (Night Map)
Notebook này được tối ưu sẵn cho GPU Kaggle (Tesla T4) để huấn luyện mô hình phân đoạn làn đường ban đêm.

In [ ]:
# 1. Kiểm tra GPU và cài đặt thư viện Ultralytics
!nvidia-smi
!pip install ultralytics -q

In [ ]:
# 2. Giải nén Dataset (nếu tải lên dạng .zip vào Kaggle Dataset)
import os, zipfile, yaml

# Kiểm tra thư mục dữ liệu đầu vào
input_path = '/kaggle/input'
print('Các dataset có sẵn trong input:', os.listdir(input_path))

# Thiết lập thư mục làm việc
DATASET_DIR = '/kaggle/working/night_yolo'
os.makedirs(DATASET_DIR, exist_ok=True)

# Tự động tìm file zip hoặc thư mục dataset đã giải nén
for root, dirs, files in os.walk(input_path):
    for f in files:
        if f.endswith('.zip'):
            zip_file = os.path.join(root, f)
            print(f'Đang giải nén: {zip_file} ...')
            with zipfile.ZipFile(zip_file, 'r') as z:
                z.extractall(DATASET_DIR)

# Nếu dataset được upload trực tiếp dạng folder:
if not os.path.exists(os.path.join(DATASET_DIR, 'images')):
    # Tìm kiếm đường dẫn images trong /kaggle/input
    for root, dirs, files in os.walk(input_path):
        if 'images' in dirs and 'labels' in dirs:
            DATASET_DIR = root
            break

print(f'Thư mục dataset sử dụng: {DATASET_DIR}')

In [ ]:
# 3. Tạo file cấu hình data.yaml cho Kaggle
yaml_config = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 1,
    'names': {0: 'road'}
}

config_file = '/kaggle/working/night_road_seg.yaml'
with open(config_file, 'w') as f:
    yaml.dump(yaml_config, f)

print(f'File cấu hình: {config_file}')
with open(config_file) as f:
    print(f.read())

In [ ]:
# 4. Khởi chạy Huấn luyện YOLO11n-seg
from ultralytics import YOLO

# Tải pretrained base model
model = YOLO('yolo11n-seg.pt')

# Train với cấu hình tối ưu cho ảnh ban đêm
results = model.train(
    data=config_file,
    epochs=60,
    imgsz=640,
    batch=16,            # Batch 16 trên GPU T4 rất nhanh và ổn định
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=20,
    hsv_v=0.4,           # Tăng cường ngẫu nhiên độ sáng/tối phục vụ sa hình đêm
    mosaic=0.5,
    fliplr=0.0,          # Tắt flip ngang để bảo toàn hướng làn đường
    project='/kaggle/working/runs',
    name='night_road_yolo11',
    save=True
)

In [ ]:
# 5. Copy file trọng số tốt nhất ra ngoài để tải về
import shutil

best_src = '/kaggle/working/runs/night_road_yolo11/weights/best.pt'
best_dst = '/kaggle/working/best_night.pt'

if os.path.exists(best_src):
    shutil.copy(best_src, best_dst)
    print(f'✅ Đã lưu trọng số tốt nhất tại: {best_dst}')
    print('Bạn có thể tải file best_night.pt từ mục Output bên phải màn hình Kaggle!')
else:
    print('Chưa tìm thấy best.pt, hãy kiểm tra thư mục runs.')